In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib



In [13]:
df = pd.read_csv("../data/processed/demographic_features.csv")

In [14]:
FEATURES = ['gender_enc',
    'age_enc',
    'membership_enc',
    'spend_enc',
    'items_enc',
    'recency_enc',
    'environment_enc']

In [16]:
# ── Sanity check ──────────────────────────────────────────
print("Label distribution:\n", df['cognitive_label'].value_counts())
print("Any NaN in features?", df[FEATURES].isna().any().any())

Label distribution:
 cognitive_label
1.0    283
0.0      1
Name: count, dtype: int64
Any NaN in features? False


In [17]:
# Drop rows where label or features are NaN
df = df.dropna(subset=FEATURES + ['cognitive_label'])

In [18]:
min_class_count = df['cognitive_label'].value_counts().min()
print(f"Smallest class has {min_class_count} samples")


Smallest class has 1 samples


In [19]:
X = df[FEATURES].values
y = df['cognitive_label'].values

# Use stratify only if every class has ≥ 2 samples
use_stratify = y if min_class_count >= 2 else None
if use_stratify is None:
    print("⚠️  Stratify disabled — too few samples in some class")


⚠️  Stratify disabled — too few samples in some class


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=use_stratify,   # ← safe stratify
    random_state=42
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 227 | Test: 57


In [21]:
# ── Train multiple demographic models ─────────────────────
models = {
    "Logistic Regression": LogisticRegression(
        class_weight='balanced', random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, random_state=42
    )
}

In [22]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='weighted')
    results[name] = {"accuracy": acc, "f1": f1, "model": model}
    print(f"\n{name}: Accuracy={acc:.4f} | F1={f1:.4f}")


Logistic Regression: Accuracy=0.9123 | F1=0.9541

Random Forest: Accuracy=1.0000 | F1=1.0000

Gradient Boosting: Accuracy=1.0000 | F1=1.0000


In [23]:
# ── Pick best demographic model ───────────────────────────
best_name = max(results, key=lambda k: results[k]['f1'])
best_demo_model = results[best_name]['model']
print(f"\nBest demographic model: {best_name}")


Best demographic model: Random Forest


In [24]:
# Save the best one Best demographic model: Random Forest
joblib.dump(best_demo_model, "../models/demographic_classifier.pkl")
print("Saved → ../models/demographic_classifier.pkl")

Saved → ../models/demographic_classifier.pkl


In [25]:
if hasattr(best_demo_model, 'feature_importances_'):
    importance = best_demo_model.feature_importances_
    for feat, imp in zip(FEATURES, importance):
        print(f"  {feat:<25} {imp:.4f}")

  gender_enc                0.0200
  age_enc                   0.0482
  membership_enc            0.0536
  spend_enc                 0.2214
  items_enc                 0.2544
  recency_enc               0.1899
  environment_enc           0.2126
